# `os` Module Exercises

These exercises will test your understanding of the `os` module, with a focus on **security**.
Make sure you run the import cell before starting.

In [ ]:
import os
import stat
import signal
import subprocess

## Exercise 1: Information Gathering

1. Print the current working directory.
2. List all files and directories in the current working directory.
3. Check if `os_intro.md` exists in the current directory.

In [ ]:
# 1. Print the current working directory
print('Current working directory:', os.getcwd())

# 2. List all files and directories
print('\nContents:', os.listdir('.'))

# 3. Check if os_intro.md exists
print('\nos_intro.md exists:', os.path.exists('os_intro.md'))
print('\nos_intro_2.md exists:', os.path.exists('os_intro_2.md'))

## Exercise 2: Directory Creation and Navigation

1. Create a directory `practice_dir`.
2. Change into it and print the new working directory.
3. Create an empty file `hello.txt` inside it.
4. Change back to the original directory.

In [ ]:
# Save the original directory
original = os.getcwd()

# 1. Create a new directory
os.mkdir('practice_dir')

# 2. Change into it
os.chdir('practice_dir')
print('Now in:', os.getcwd())

# 3. Create an empty file
open('hello.txt', 'w').close()
print('Files here:', os.listdir('.'))

# 4. Go back to the original directory
os.chdir(original)
print('Back in:', os.getcwd())

## Exercise 3: File Renaming and Deletion

1. Rename `practice_dir/hello.txt` to `practice_dir/goodbye.txt`.
2. Check if `goodbye.txt` exists.
3. Delete the file, then the directory.

In [ ]:
# 1. Rename the file
os.rename('practice_dir/hello.txt', 'practice_dir/goodbye.txt')

# 2. Verify it exists
print('goodbye.txt exists:', os.path.exists('practice_dir/goodbye.txt'))

# 3. Delete the file, then the directory
print('Removing goodbye.txt...')
os.remove('practice_dir/goodbye.txt')
print('goodbye.txt exists:', os.path.exists('practice_dir/goodbye.txt'))

print('Removing practice_dir...')
os.rmdir('practice_dir')
print('practice_dir exists:', os.path.exists('practice_dir'))

## Exercise 4: Path Manipulation

Given `['home', 'user', 'documents', 'projects']`:
1. Use `os.path.join` to build a single path.
2. Use `os.path.split` to separate the last component.
3. Print `os.path.dirname` and `os.path.basename`.

In [ ]:
# 1. Build a path from parts
parts = ['home', 'user', 'documents', 'projects']
path = os.path.join(*parts)
print('Joined path:', path)

# 2. Split the path
# Note: os.path.split() splits the path into head and tail, where tail is the last component
# and head is everything before it.
head, tail = os.path.split(path)
print(f'Split -> head: {head}, tail: {tail}')

# 3. dirname and basename
print('dirname:', os.path.dirname(path))
print('basename:', os.path.basename(path))

## Exercise 5: Environment Variables

1. Print the `PATH` environment variable.
    - `PATH` contains a set of directories where executable programs are located.
2. Get `MY_FAKE_VAR` with default `'Not Found'`.
3. Set `COURSE_NAME` to `'CybSec OS Module'` and print it.

In [ ]:
# 1. Print PATH
print('PATH:', os.getenv('PATH'))

# 2. Get a non-existent variable with a default
# os.getenv() returns None if the variable is not found, but we can provide a default value to return instead (i.e., 'Not Found').
print('MY_FAKE_VAR:', os.getenv('MY_FAKE_VAR', 'Not Found'))

# 3. Set and print a new variable
os.environ['COURSE_NAME'] = 'CybSec OS Module'
print('COURSE_NAME:', os.environ['COURSE_NAME'])

## Exercise 6: Walking a Directory Tree

Run the setup cell, then use `os.walk()` to print all files and directories in `test_tree`.

In [ ]:
# Setup
# os.makedirs() creates directories recursively. The exist_ok=True argument prevents an error if the directory already exists.
os.makedirs('test_tree/folder_a', exist_ok=True)
os.makedirs('test_tree/folder_b/subfolder', exist_ok=True)

open('test_tree/file1.txt', 'w').close()
open('test_tree/folder_a/file2.txt', 'w').close()
open('test_tree/folder_b/subfolder/file3.csv', 'w').close()

print('Setup complete!')

In [ ]:
for dirpath, dirnames, filenames in os.walk('test_tree'):
    print(f'\nDirectory: {dirpath}')
    for d in dirnames:
        print(f'  Subdir: {d}')
    for f in filenames:
        print(f'  File: {f}')

## Exercise 7: Process Information

1. Print the current process ID (`os.getpid()`) and the parent process ID (`os.getppid()`).
2. Use `subprocess.run` to execute `ps -p <your_pid> -o pid,ppid,comm` and display the output.
3. Use `subprocess.run` to execute `whoami` and capture its output.

In [ ]:
# 1. Get current process info
pid = os.getpid()
gid = os.getgid()
uid = os.getuid()
log = os.getlogin()

ppid = os.getppid()
print(f'Current process ID: {pid}')
print(f'Current group ID: {gid}')
print(f'Current user ID: {uid}')
print(f'Current login name: {log}')
print(f'Parent process ID:  {ppid}')

# 2. Use ps to display process info to verify PID and PPID
result = subprocess.run(
    ['ps', '-p', str(pid), '-o', 'pid,ppid,comm'],
    capture_output=True, text=True
)
print('\nps output:')
print(result.stdout)

# 3. Capture output of whoami
result = subprocess.run(['whoami'], capture_output=True, text=True)
print(f'I am: {result.stdout.strip()}')

## Exercise 8: Security — Command Injection

This exercise demonstrates **why `os.system()` is dangerous**.

1. Define a variable `user_input = 'hello; echo HACKED'`.
    - `echo` is a command that writes text on the standard output.
2. Run `os.system('echo ' + user_input)`.
    - Observe how the injected command runs.
3. Now do the same with `subprocess.run(['echo', user_input])`.
    - Observe how the injection is **neutralized**.
4. Explain in a comment why `subprocess` with a list is safer.

In [ ]:
user_input = 'hello; echo HACKED'

# VULNERABLE: the shell interprets the semicolon as a command separator
# This allows an attacker to execute arbitrary commands by injecting them into the input.
print('--- os.system (VULNERABLE) ---')
os.system('echo ' + user_input)

# SAFE: arguments are passed as a list, no shell interpretation
print('\n--- subprocess.run (SAFE) ---')
result = subprocess.run(['echo', user_input], capture_output=True, text=True)
print(result.stdout)

# Explanation:
# subprocess.run() with a list does NOT invoke a shell.
# Special characters like ; | & are treated as literal characters,
# not as shell operators. This prevents command injection attacks.

## Exercise 9: Security — File Permissions

1. Create a file `ssh_key.pem`.
2. Print its default permissions using `oct(os.stat(...).st_mode)`.
3. Set permissions to owner-read-only (`0o400`) with `os.chmod`.
4. Verify permissions changed.
5. Try to open the file for writing — what happens?

In [ ]:
# 1. Create a dummy SSH key file
open('ssh_key.pem', 'w').close()

# 2. Check default permissions
print('Default permissions:', oct(os.stat('ssh_key.pem').st_mode))

# Three octal digits represent permissions for owner, group, and others.
# Each digit is a sum of 4 (read), 2 (write), and 1 (execute).
print('Alternative view:   ', stat.filemode(os.stat('ssh_key.pem').st_mode))

# 3. Restrict to owner-read-only
os.chmod('ssh_key.pem', 0o400)

# 4. Verify the change
print('\nAfter chmod 400: ', oct(os.stat('ssh_key.pem').st_mode))
print('Alternative view:', stat.filemode(os.stat('ssh_key.pem').st_mode))

# 5. Try to write to it — should raise PermissionError
try:
    with open('ssh_key.pem', 'w') as f:
        f.write('\nshould fail')
except PermissionError as e:
    print(f'\nExpected error: {e}')

# Cleanup
os.chmod('ssh_key.pem', 0o644)
os.remove('ssh_key.pem')

## Exercise 10: Security — Environment Variable Leak Detection

Secrets (API keys, passwords) are often stored in environment variables.
Dumping `os.environ` carelessly can leak credentials.

1. Set `os.environ['AWS_SECRET_ACCESS_KEY'] = 'SuperSecret123!'`.
2. Set `os.environ['DB_PASSWORD'] = 'p@ssw0rd'`.
3. Iterate over `os.environ.keys()`:
    - if a key contains `SECRET`, `PASSWORD`, or `KEY` (case-insensitive), print a **warning** but **NOT** the value;
    - otherwise, just print the key name.

In [ ]:
# 1 & 2. Set dummy secrets
os.environ['AWS_SECRET_ACCESS_KEY'] = 'SuperSecret123!'
os.environ['DB_PASSWORD'] = 'p@ssw0rd'

# 3. Iterate and detect sensitive variables
sensitive_keywords = ['SECRET', 'PASSWORD', 'KEY', 'TOKEN']

for key in sorted(os.environ.keys()):
    if any(kw in key.upper() for kw in sensitive_keywords):
        print(f'\u26a0\ufe0f  SENSITIVE: {key} = [HIDDEN]')
    else:
        print(key)

## Exercise 11: Security — Detecting SUID/SGID Binaries

SUID (Set User ID) binaries run with the permissions of the file owner, not the caller.
Attackers often look for SUID-root binaries to escalate privileges.

Write a script that walks `/usr/bin` and lists any file with the SUID or SGID bit set.
Use `stat.S_ISUID` and `stat.S_ISGID` constants with bitwise AND on `st_mode`.

In [ ]:
target_dir = '/usr/bin'
print(f'Scanning {target_dir} for SUID/SGID binaries...\n')

count = 0
for entry in os.listdir(target_dir):
    full_path = os.path.join(target_dir, entry)
    if not os.path.isfile(full_path):
        continue
    try:
        st = os.stat(full_path)
        flags = []
        if st.st_mode & stat.S_ISUID:
            flags.append('SUID')
        if st.st_mode & stat.S_ISGID:
            flags.append('SGID')
        if flags:
            count += 1
            print(f'  {full_path} -> {" | ".join(flags)}  (mode: {oct(st.st_mode)})')
    except PermissionError:
        pass

print(f'\nTotal SUID/SGID binaries found: {count}')

---
# Homework Assignments

### Homework 1: Permission Security Auditor

Write a function `audit_permissions(path)` that walks a directory tree and reports:
- Files that are **world-writable** (`stat.S_IWOTH`).
- Files that are **world-readable** (`stat.S_IROTH`) AND contain sensitive extensions (`.pem`, `.key`, `.env`, `.conf`).
- Files with **SUID** bit set.

Print the absolute path and the issue found for each flagged file.
Test with the setup cell below.

In [ ]:
# Setup
import shutil
shutil.rmtree('audit_dir', ignore_errors=True)
os.makedirs('audit_dir/project1', exist_ok=True)
os.makedirs('audit_dir/project2', exist_ok=True)
files_setup = {
    'audit_dir/project1/safe_config.txt': 0o644,
    'audit_dir/project1/vulnerable_script.sh': 0o777,
    'audit_dir/project2/secret.key': 0o644,
    'audit_dir/project2/database.env': 0o666,
    'audit_dir/project2/notes.txt': 0o600,
}
for f, mode in files_setup.items():
    open(f, 'w').close()
    os.chmod(f, mode)
print('Audit setup complete!')

In [ ]:
def audit_permissions(path):
    # Your code here
    pass

audit_permissions('audit_dir')

### Homework 2: Simple Process Monitor

Write a script that:
1. Uses `subprocess.run` to execute `ps aux` and capture the output.
2. Parses each line and extracts the **user**, **PID**, and **command**.
3. Prints only the processes owned by `root`.
4. Counts how many root processes are running.

In [ ]:
# Your code here


### Homework 3: File Organizer

Write a script that organizes a messy folder by file extension.
1. Run the setup cell.
2. For each file, get its extension with `os.path.splitext()`.
3. Create a subfolder named `<ext>_files` if it doesn't exist.
4. Move the file into that subfolder using `os.rename()`.

In [ ]:
# Setup
import shutil
shutil.rmtree('messy_folder', ignore_errors=True)
os.makedirs('messy_folder', exist_ok=True)
for f in ['doc1.txt', 'doc2.txt', 'img.jpg', 'img2.jpg', 'data.csv', 'data2.csv']:
    open(os.path.join('messy_folder', f), 'w').close()
print('Messy folder created!')

In [ ]:
# Your code here


---
### Cleanup
Run this cell to remove all temporary files created during the exercises.

In [ ]:
import shutil
for d in ['test_tree', 'audit_dir', 'messy_folder', 'practice_dir']:
    shutil.rmtree(d, ignore_errors=True)
for f in ['ssh_key.pem']:
    if os.path.exists(f):
        os.remove(f)
print('Cleanup done!')